In [5]:
# extract phenotypes
library(de)
library(data.table)
library(tidyverse)
library(ggrepel)
library(lubridate)
#install.packages('bit64')
setwd('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')

ERROR: Error in library(devtools): there is no package called 'devtools'


In [4]:
#devtools::load_all('utils/modules/R/ukbtools/ukbtools.Rproj')

ERROR: Error in loadNamespace(x): there is no package called 'devtools'


In [18]:
# phenotype tables
imp <- list(
    phenotypes = "/well/lindgren/UKBIOBANK_DATA_LINDGREN/Phenotype_data/October2017/ukb10844.csv", # october 2018 release
    sample_genetic_eur = "/well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt",
    sample_sex = "/well/lindgren/UKBIOBANK/nbaya/resources/ukb11867_sex.tsv",
    sample_qc = "/well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt",
    sample_fam = "/well/lindgren/UKBIOBANK/DATA/SAMPLE_FAM/ukb11867_cal_chr1_v2_s488363.fam", # family and indiviudal IDs
    sample_excl = "/well/lindgren/UKBIOBANK/ferreira/SAMPLE_QC/ukb10844_QCexclIDs_TF_21082020", #General QC - exclusion IDs (created in March 2020 by TF; see hormones project for criteria(, including individuals that withdrew consent as of Aug/2020)
    pcs = "/gpfs3/well/lindgren/UKBIOBANK/nbaya/resources/ukb_40pcs.txt.gz"
)

# 
wes <- list(
    phenotypes = "/well/lindgren/UKBIOBANK_DATA_LINDGREN/Phenotype_data/October2017/ukb10844.csv", # october 2018 release
    sample_genetic_eur = "/well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt",
    sample_sex = "/well/lindgren/UKBIOBANK/nbaya/resources/ukb11867_sex.tsv",
    sample_qc = "/well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt",
    sample_fam = "/well/lindgren/UKBIOBANK/DATA/WES_200k/ukb23155_c1_b0_v1_s200632.fam", # family and indiviudal IDs
    sample_excl = "/well/lindgren/UKBIOBANK/ferreira/SAMPLE_QC/ukb10844_QCexclIDs_TF_21082020", #General QC - exclusion IDs (created in March 2020 by TF; see hormones project for criteria(, including individuals that withdrew consent as of Aug/2020)
    pcs = "/gpfs3/well/lindgren/UKBIOBANK/nbaya/resources/ukb_40pcs.txt.gz"
)

# conversion tables
sample_conversion <- "/well/lindgren/UKBIOBANK/DATA/WES/bridge_11867_12788.csv" # eid_11867, eid_12788
icd_conversion <- "/well/lindgren/UKBIOBANK/ferreira/convert_disease_codes/icd9_icd10_lkps.txt"

In [19]:
# 
#validate_list_paths <- function(lst){
#    files_found = unlist(lapply(lst, file.exists))
#    paths_not_found = paste0(unlist(lst)[!files_found], collapse = '\n')
#    if (any(!files_found)) stop(paste0('The following paths does not exists:\n',paths_not_found))
#    return(invisible(TRUE))
#}
#
"%nin%" <- function(x, y) !(x %in% y)

In [20]:
#' calculate time since
calc_time_since <- function(time, age_day = lubridate::today(), units = "years", floor = TRUE) {
  the_age = lubridate::interval(time, age_day) / lubridate::duration(num = 1, units = units)
  if (floor) return(as.integer(floor(the_age)))
  return(the_age)
}


In [21]:
#dt = fread(wes$phenotypes, header = TRUE)

#obj <- imp

# how many individuals in the FAM file is in the Pheno file?
#dt <- fread(obj$phenotypes)
#dt_sample_fam <- fread(obj$sample_fam)
#n_found <- sum(dt_sample_fam[[1]] %in% dt$eid)
#n_not_found <- sum(dt_sample_fam[[1]] %nin% dt$eid)
#write(paste('*',n_not_found,'of',n_found,'individuals could not be mapped.'),stdout())

In [22]:
obj <- imp
dt <- fread(obj$phenotypes, nrows = 100)

In [23]:
# setup core feature matrix
dt_sample_fam <- fread(obj$sample_fam)
dt_genetic_eur <- fread(obj$sample_genetic_eur)
dt_sample_qc <- fread(obj$sample_qc)


In [24]:
#dt <- head(d1, n = 100)

In [25]:
colnames(dt) <- gsub('-','.',colnames(dt))
colnames(dt)[colnames(dt) != 'eid'] <- paste0('X', colnames(dt)[colnames(dt) != 'eid'])

In [40]:
## sample level QC and standard phenotypes
# read exclusion samples
# add ID to ukb_sqc_v2.txt using ukb11867_cal_chr1_v2_s488363.fam 
# use ukb_sqc_v2.txt to add submitted gender
# use ukb_sqc_v2.txt to add inferred gender
# use ukb_sqc_v2.txt to add white british ancestry
# tally up current men/women/white british

# add recruitment age
# encode genotyping array using "X22000.0.0"
# encode ukbb.centre using "X54.0.0""
# add PCs using "22009" (with grep?)
# add ID using 'f.eid'

# age encodings (#Year of birth (year 34-0.0, month 52-0.0))
# encode age using "X53.0.0"
# encode age squared using "X53.0.0"
# encode age cubed using "X53.0.0"

## phenotype processing
# open ICD10-ICD9 code bridge file: "/well/lindgren/UKBIOBANK/ferreira/phenotype_extraction/phenotype_lists/ICD10_codes_converted_ICD9_v2v3_codes.txt
# get phenotype data from before
# get primary care data from: "/well/lindgren/UKBIOBANK/DATA/PHENOTYPE/PRIMARY_CARE/gp_clinical.txt"


In [26]:
extract_covars <- function(dt_pheno, dt_sample_qc, keep = 'eid'){
    
    # combine dt and dt_sample_qc
    #keep = c('eid','Submitted.Gender','Inferred.Gender','in.white.British.ancestry.subset')
    remove <- colnames(dt_pheno)[!colnames(dt_pheno) %in% keep]
    dt_sample_qc$eid <- dt_sample_fam$V1
    dt_sample_qc <- dt_sample_qc[,c('eid','Submitted.Gender','Inferred.Gender','in.white.British.ancestry.subset')]
    sum(dt_pheno$eid %in% dt_sample_qc$eid)
    dt <- merge(dt_pheno, dt_sample_qc)

    # get data of birth
    dt$yob <- dt$X34.0.0
    dt$mob <- dt$X52.0.0
    dt$dob <- as.Date(paste0(dt$yob,'-',dt$mob,'-',1))

    # get date of attending recruitment centre
    dt$centre_attend0 <- as.Date(dt$X53.0.0)
    dt$centre_attend1 <- as.Date(dt$X53.1.0)
    dt$centre_attend2 <- as.Date(dt$X53.2.0)

    # get age when attending recruitment centre
    dt$centre_attend_age0 <- calc_time_since(dt$dob, dt$centre_attend0)
    dt$centre_attend_age1 <- calc_time_since(dt$dob, dt$centre_attend1)
    dt$centre_attend_age2 <- calc_time_since(dt$dob, dt$centre_attend2)

    # get age to be used as covariates
    dt$age <- dt$centre_attend_age0
    dt$age1 <- dt$centre_attend_age0^2
    dt$age2 <- dt$centre_attend_age0^3

    # get principal components 
    bool_pc <- grep(22009,colnames(dt))
    n <- length(colnames(dt)[bool_pc])
    colnames(dt)[bool_pc] <- paste0('PC',1:n)

    # negative coding from BiLEVE (recoded 1); positive coding from axiom (recoded 2)
    dt$genotyping.array <- ifelse(dt$X22000.0.0 <0, 1, 2)
    dt$sex = ifelse(dt$Inferred.Gender == "F", 2, ifelse(dt$Inferred.Gender == "M", 1, NA))
    colnames(dt)[colnames(dt) == "X54.0.0"] = "ukbb.centre"
    
    # clean up
    bool_keep <- ! colnames(dt) %in% remove
    dt = dt[,bool_keep, with = FALSE]
    
    return(dt)
}

In [27]:
remove <- colnames(dt)[!colnames(dt) %in% 'eid']
c('age') %in% remove
q <- extract_covars(dt, dt_sample_qc)
q

[1] FALSE

eid,ukbb.centre,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,⋯,centre_attend1,centre_attend2,centre_attend_age0,centre_attend_age1,centre_attend_age2,age,age1,age2,genotyping.array,sex
<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<date>,<date>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
1012978,11005,-14.22390,4.316110,-2.583720,4.110260,13.956500,1.5437400,2.5340200,4.2530500,⋯,NA,NA,51,NA,NA,51,2601,132651,2,2
1025876,11003,-10.88470,2.597800,-3.181410,-2.436430,-8.347310,0.2943190,1.3257300,1.9210100,⋯,NA,NA,41,NA,NA,41,1681,68921,2,2
1026769,11009,-11.73770,4.467900,-0.870203,0.495115,-2.478190,-2.9397400,-2.3961400,-1.5207000,⋯,NA,NA,66,NA,NA,66,4356,287496,2,2
1040495,11013,-11.94830,5.276710,0.114861,-5.288530,-4.577080,-2.9125600,5.6781700,-0.6177910,⋯,NA,NA,66,NA,NA,66,4356,287496,2,1
1052795,11020,-8.21625,3.971510,-1.450520,-9.344880,-5.281080,-0.9078920,2.0279400,1.7880100,⋯,NA,NA,59,NA,NA,59,3481,205379,2,1
1105858,11009,-8.85043,4.623670,1.649860,2.622890,8.842090,-1.4098900,1.5634800,3.6415500,⋯,NA,NA,49,NA,NA,49,2401,117649,2,2
1141098,11010,-12.89680,4.037370,-2.344310,2.512150,-6.746900,-1.0318400,-0.2375480,-3.3133400,⋯,NA,NA,58,NA,NA,58,3364,195112,2,2
1211428,11021,352.54000,69.735200,-4.430110,4.055590,0.879280,2.5685300,-0.7522970,-0.0639162,⋯,NA,NA,50,NA,NA,50,2500,125000,2,1
1225243,11013,-9.84792,2.461370,-0.705939,2.169720,5.664680,0.7055550,0.3443690,-2.1653300,⋯,NA,NA,48,NA,NA,48,2304,110592,2,2


In [28]:
make_icd_mapping <- function(from, to, icd_path = "/well/lindgren/UKBIOBANK/ferreira/convert_disease_codes/icd9_icd10_lkps.txt"){
    
    # Read icd code
    icd <- fread(icd_path)
    colnames(icd) <- gsub('_code','',colnames(icd))
    stopifnot(c(from,to) %in% colnames(icd))
    
    # Deal with NAs
    icd <- icd[,colnames(icd) %in% c(from, to),with = F]
    icd <- icd[!duplicated(icd)]
    icd[icd==''] <- NA
    icd[icd=='UNDEF'] <- NA
    icd <- icd[!is.na(icd[[from]]), ]
    mapping <- as.list(icd[[to]])
    names(mapping) <- icd[[from]]

    # return function that can do the mapping
    mapper <- function(code){
        map <- mapping[code]
        ids <- names(map)
        is_na <- is.na(ids)
        names(map) <- code
        map[is_na] <- NA
        return(as.character(unlist(map)))
    }
    
    return(mapper)
}








[1] NA                             "Trachea_bronchus_lung_cancer"

In [14]:
# count gender / white british
out_df <- as.data.frame(table(mrg$Inferred.Gender, mrg$in.white.British.ancestry.subset))
colnames(out_df) <- c('sex','wb','count')
out_df

ERROR: Error in table(mrg$Inferred.Gender, mrg$in.white.British.ancestry.subset): object 'mrg' not found


In [ ]:
#dt[['X22000.0.0']]
colnames(dt)[grepl('22000',colnames(dt))]

In [30]:
gp=fread("/well/lindgren/UKBIOBANK/DATA/PHENOTYPE/PRIMARY_CARE/gp_clinical.txt", header=T, sep="\t", nrows = 100) 

In [55]:
dt_biomarker <- fread('/gpfs1/well/lindgren/UKBIOBANK/DATA/Biomarker_data/ukb27722.csv', nrows = 1000 )

Warning message in require_bit64_if_needed(ans):
"Some columns are type 'integer64' but package bit64 is not installed. Those columns will print as strange looking floating point data. There is no need to reload the data. Simply install.packages('bit64') to obtain the integer64 print method and print the data again."


Warning message in require_bit64_if_needed(ans):
"Some columns are type 'integer64' but package bit64 is not installed. Those columns will print as strange looking floating point data. There is no need to reload the data. Simply install.packages('bit64') to obtain the integer64 print method and print the data again."


In [104]:
find_data_fields <- function(traits='chrons', by = 'coding_description'){
    
    #m <- fread('../utils/modules/R/ukbtools/extdata/pan_ukbb_manifest_210916.csv')
    stopifnot(by %in% colnames(m))
    
    
    
    lapply(traits, function(tr) m[ m[[by]] == tr,])
    
    
}


find_data_fields()




Warning message in cbind(parts$left, ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"


trait_type,phenocode,pheno_sex,coding,modifier,description,description_more,coding_description,category,n_cases_full_cohort_both_sexes,,filename,filename_tabix,aws_link,aws_link_tabix,wget,wget_tabix,size_in_bytes,size_in_bytes_tabix,md5_hex,md5_hex_tabix
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int64>,<int>,<chr>,<chr>


In [131]:
manifest_search_trait <- function(traits, by = 'coding_description', manifest_file = '/utils/modules/R/ukbtools/extdata/pan_ukbb_manifest_210916.csv'){
    stopifnot(by %in% colnames(m))
    search_matrix <- do.call(rbind, lapply(traits, function(tr) { 
        col = tolower(m[[by]])
        bool_trait = grepl(tr, col)
        bool_not = grepl('not', col)
        m = m[bool_trait & !bool_not,]
        m$search_term = tr
        return(m)
    }))
    return(search_matrix)
}


In [153]:

cancer_manifest <- manifest_search_trait('bladder +cancer')



#field_code_summary

cdt <- data.frame(origin = colnames(dt))
cdt$phenocode <- NA
phenocode_bool <- grepl('X', cdt$origin)
phenocodes <- strsplit( phenocodes[phenocode_bool], split = "X|\\.")
#cdt$phenocode[phenocode_bool] <- 
#phenocodes

# Look for trait we want to investigate
#trait = 'cancer'
#bool = grepl(trait, m$coding_description)
#cur_phenocode = m[bool,]$phenocode
#print(cur_phenocode)


# code.time_of_assesment.number_of_values
#'X20001.0.0'


#dt_bool = colnames(dt) == 'X20001.0.0'
#coding = as.vector(na.omit(dt[,dt_bool, with = F]))





ERROR: Error in strsplit(phenocodes[phenocode_bool], split = "X|\\."): non-character argument


In [69]:
#sum(grepl('30640',colnames(dt_biomarker)))

bool = grepl('1002', colnames(dt))
colnames(dt)[bool]


[1] "X21002.0.0"  "X21002.1.0"  "X21002.2.0"  "X100200.0.0" "X100200.1.0"
 [6] "X100200.2.0" "X100200.3.0" "X100200.4.0" "X100210.0.0" "X100210.1.0"
[11] "X100210.2.0" "X100210.3.0" "X100210.4.0" "X100220.0.0" "X100220.1.0"
[16] "X100220.2.0" "X100220.3.0" "X100220.4.0" "X100230.0.0" "X100230.1.0"
[21] "X100230.2.0" "X100230.3.0" "X100230.4.0" "X100240.0.0" "X100240.1.0"
[26] "X100240.2.0" "X100240.3.0" "X100240.4.0" "X100250.0.0" "X100250.1.0"
[31] "X100250.2.0" "X100250.3.0" "X100250.4.0" "X100260.0.0" "X100260.1.0"
[36] "X100260.2.0" "X100260.3.0" "X100260.4.0" "X100270.0.0" "X100270.1.0"
[41] "X100270.2.0" "X100270.3.0" "X100270.4.0" "X100280.0.0" "X100280.1.0"
[46] "X100280.2.0" "X100280.3.0" "X100280.4.0" "X100290.0.0" "X100290.1.0"
[51] "X100290.2.0" "X100290.3.0" "X100290.4.0"

In [45]:
icd10 <- make_icd_mapping('ICD10','DESCRIPTION_ICD10')
icd9_icd10 <- make_icd_mapping('ICD9','ICD10')
icd9 <- make_icd_mapping('ICD9','DESCRIPTION_ICD9')

In [33]:
m <- dt
cn <- colnames(m)
bool <- grepl('41202',cn)
print(sum(bool))
m[,bool, with = F]


[1] 380


X41202.0.0,X41202.0.1,X41202.0.2,X41202.0.3,X41202.0.4,X41202.0.5,X41202.0.6,X41202.0.7,X41202.0.8,X41202.0.9,⋯,X41202.0.370,X41202.0.371,X41202.0.372,X41202.0.373,X41202.0.374,X41202.0.375,X41202.0.376,X41202.0.377,X41202.0.378,X41202.0.379
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
,,,,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
M169,,,,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
N390,N133,K296,K259,D122,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
O049,K660,N946,J329,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
K083,,,,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
H830,K409,,,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
R001,I48,,,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
L721,,,,,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
R31,R194,N40,K573,N459,,,,,,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [54]:
icd10(c('M169','I48','J329'))

[1] "Coxarthrosis, unspecified"      NA                              
[3] "Chronic sinusitis, unspecified"

In [59]:
#dt_biomarker[,grepl('40006',cn), with = F]

In [113]:
# for each "disease"
# extract ICD10 code
# use dt_biomarker file to extract relevant disease using ICD10 code

test <- fread(icd_conversion)
length(unique(test$ICD10))

[1] 219

In [90]:
disease <- make_icd_mapping('disease','ICD10')


In [101]:
disease('breast_cancer')

[1] "C50"

In [99]:
length(unique(test$disease))
unique(test$disease)

[1] 37

[1] "colorectal_cancer"                    
 [2] "Trachea_bronchus_lung_cancer"         
 [3] "breast_cancer"                        
 [4] "hypothalamic_amenorrhea"              
 [5] "PCOS"                                 
 [6] "POI"                                  
 [7] "dementia"                             
 [8] "Alzheimers_disease"                   
 [9] "depression"                           
[10] "ADHD"                                 
[11] "renal_failure"                        
[12] "coronary_artery_disease"              
[13] "ischaemic_heart_disease"              
[14] "stroke"                               
[15] "stroke_hemorrhagic"                   
[16] "ischaemic_stroke"                     
[17] "chronic_obstructive_pulmonary_disease"
[18] "Crohns_disease"                       
[19] "IBD"                                  
[20] "Cirrhosis"                            
[21] "NASH"                                 
[22] "NAFLD"                                
[23] "psoriais"                             
[24] "hyperandrogenism"                     
[25] "hematuria"                            
[26] "proteinuria"                          
[27] "acute_renal_failure"                  
[28] "chronic_kidney_disease"               
[29] "male_infertility"                     
[30] "oligomenorrhea"                       
[31] "habitual_aborter"                     
[32] "female_infertility"                   
[33] "ectopic_pregnancy"                    
[34] "Preeclampsia"                         
[35] "GDM"                                  
[36] "intrahepatic_cholestasis_in_pregnancy"
[37] "polycystic_kidney_disease"

In [106]:
'N133' %in% test$V2_code
'N133' %in% test$V2_code
'N133' %in% test$V3_code

[1] FALSE

[1] FALSE

In [30]:
colnames(x)

[1] "ICD9"              "DESCRIPTION_ICD9"  "ICD10"            
[4] "DESCRIPTION_ICD10"